# Laptop Price Prediction — Exploratory Data Analysis

## Objective

The objective of this notebook is to perform Exploratory Data Analysis (EDA) on the laptop dataset and understand the factors that influence laptop prices.

We will analyze:

- Dataset structure
- Data types
- Missing values
- Duplicate records
- Numerical features
- Categorical features
- Price distribution
- Outliers
- Feature relationships
- Correlations

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/raw/laptops.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='str')

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique().sort_values()

## Initial Observations

Based on the initial inspection of the dataset:

- The dataset contains 991 rows and 22 columns. 
- The target variable is `Price`.
- No missing values are present in dataset.
- No duplicate records were found.
- The `Model` column contains 991 unique laptop models.
- Further analysis is required to identify outliers and relationships between features and price.

-----------------------------------------------------------------------------------------------




## Target Variable Analysis

The target variable for this project is `Price`.

We will analyze:

- Minimum and maximum price
- Mean and median price
- Price distribution
- Possible skewness
- Potential outliers

In [ ]:
df['Price'].describe()

In [ ]:
df["Price"].skew()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    x = df['Price'],
    bins = 30,
    kde=True
)
plt.title("Distribution of Laptop prices")

plt.show()

In [ ]:
plt.figure(figsize=(10,4))

sns.boxplot(
    data=df, 
    x='Price'
)
plt.title("Box Plot of laptop prices")
plt.xlabel("Price (₹)")

plt.show()

### Observations

- The laptop price distribution is strongly right-skewed.
- Most laptops are concentrated in the lower-to-middle price range.
- The box plot shows several observations beyond the upper whisker.
- These high-priced laptops may represent genuine premium or high-performance devices, so they should not be removed without further investigation.
- The skewed distribution should be considered during model development and evaluation.

--------------------------------------------------------------------------------------------------------------------------

## Numerical Feature Analysis

Numerical features describe measurable laptop characteristics such as RAM, processor cores, storage capacity, display size, and resolution.

We will examine their distributions to identify their ranges, patterns, and potential anomalies.

In [ ]:
numeric_features = df.select_dtypes(include=np.number).columns

numeric_features

In [ ]:
numeric_features= df.select_dtypes(include=np.number).columns.drop(
    ["index", "Price"]
)

numeric_features

In [ ]:
df[numeric_features].describe().T

In [ ]:
df[numeric_features].hist(
    figsize=(15, 12),
    bins=20,
)
plt.tight_layout()
plt.show()


## Correlation Analysis

Correlation analysis helps us understand the linear relationship between numerical features and the target variable, `Price`.

We will examine the correlation between the numerical features and identify which features have stronger or weaker relationships with laptop prices.

In [ ]:
correlation = df[numeric_features.tolist() + ["Price"]].corr()

correlation["Price"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix of Numerical Features")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

price_corr = correlation["Price"].drop("Price").sort_values()

price_corr.plot(kind='barh')

plt.title("Correlation of Numerical Features with Price")
plt.xlabel("Correlation")
plt.ylabel("Feature")

plt.show()

-----------------------------------



## Categorical Feature Analysis

Categorical features represent groups or categories of laptop specifications.

We will analyze their value distributions and investigate how different categories are associated with laptop prices.

In [ ]:
# exclude=np.number is more robust than listing dtype names (works regardless
# of whether text columns are stored as 'object' or 'str' dtype)
categorical_features = df.select_dtypes(exclude=np.number).columns

categorical_features

In [ ]:
categorical_features = categorical_features.drop("Model")
categorical_features

In [ ]:
for col in categorical_features:
    print(df[col].value_counts())
    print()

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, col in zip(axes, categorical_features):
    
    sns.countplot(
        data=df,
        x=col,
        ax=ax
    )
    
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=75)
    
plt.tight_layout()
plt.show()

## Categorical Features vs. Price

Now that we've seen how categorical features are distributed on their own, we look at how each one relates to `Price`. This helps identify which categories (e.g. specific brands, processor tiers, GPU types) tend to command higher or lower prices.

`brand` has many unique values, so we focus on the top 10 most common brands to keep the plot readable. The remaining categorical features are plotted in a grid, each ordered by median price for easier comparison.

In [ ]:
top_brands = df['brand'].value_counts().nlargest(10).index

brand_order = (
    df[df['brand'].isin(top_brands)]
    .groupby('brand')['Price']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df[df['brand'].isin(top_brands)],
    x='brand',
    y='Price',
    order=brand_order
)

plt.title("Price Distribution by Brand (Top 10 Most Common)")
plt.xlabel("Brand")
plt.ylabel("Price (₹)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
other_categorical = categorical_features.drop('brand')

fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(18, 18)
)

axes = axes.flatten()

for ax, col in zip(axes, other_categorical):

    order = (
        df.groupby(col)['Price']
        .median()
        .sort_values(ascending=False)
        .index
    )

    sns.boxplot(
        data=df,
        x=col,
        y='Price',
        order=order,
        ax=ax
    )

    ax.set_title(f"Price by {col}")
    ax.tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

## EDA Summary

- `Price` is right-skewed, with Apple laptops priced noticeably higher and budget brands (Infinix, Zebronics) at the low end.
- Processor tier, RAM, storage, GPU type, and resolution all show a clear relationship with price — higher-end specs mean higher prices.
- No missing values or duplicates, but `index` and `Model` need handling before modeling.
- These insights will guide feature selection and engineering next.

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isnull().sum(),
    "unique": df.nunique(),
    "duplicates": df.duplicated().sum()
})

summary